In [1]:
import cv2
from deepface import DeepFace

# 1. Load the Haar Cascade face detector
face_cascade_path = (
    cv2.data.haarcascades
    + "haarcascade_frontalface_default.xml"
)

face_cascade = cv2.CascadeClassifier(face_cascade_path)

if face_cascade.empty():
    raise RuntimeError("Could not load the Haar Cascade classifier.")

# 2. Open the webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

print("Emotion detection started. Press 'q' to quit.")

try:
    while True:
        # 3. Capture a video frame
        ret, frame = cap.read()

        if not ret:
            print("Could not read webcam frame.")
            break

        # 4. Convert frame to grayscale for face detection
        gray_frame = cv2.cvtColor(
            frame, cv2.COLOR_BGR2GRAY
        )

        gray_frame = cv2.equalizeHist(gray_frame)

        # 5. Detect faces
        faces = face_cascade.detectMultiScale(
            gray_frame,
            scaleFactor=1.05,
            minNeighbors=3,
            minSize=(30, 30)
        )

        # 6. Analyze each detected face
        for (x, y, w, h) in faces:
            face_roi = frame[y:y+h, x:x+w]

            try:
                result = DeepFace.analyze(
                    face_roi,
                    actions=["emotion"],
                    enforce_detection=False,
                    detector_backend="skip"
                )

                emotion = result[0]["dominant_emotion"]

                # Draw face rectangle
                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 0, 255),
                    2
                )

                # Display predicted emotion
                cv2.putText(
                    frame,
                    emotion,
                    (x, max(y - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.9,
                    (0, 0, 255),
                    2
                )

            except Exception as e:
                print("Emotion analysis failed:", e)

        # 7. Display the live video
        cv2.imshow(
            "Real-Time Facial Emotion Recognition",
            frame
        )

        # 8. Press q to exit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    # 9. Always release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released. Program stopped.")



Emotion detection started. Press 'q' to quit.
Webcam released. Program stopped.
